# 04 — Final fitting, calibration and evaluation

This notebook fits the selected GBT-B configuration on all valid 2017–2022 training windows, fits isotonic calibration on the 2023 calibration partition, and evaluates the final predictor on 2024. Calibrated probabilities at the six anchor thresholds are projected with PAVA to enforce non-increasing exceedance probability, with linear interpolation between adjacent anchors.

In [ ]:
import json
import os
import random
import subprocess
import sys
import tempfile
import time
from functools import reduce
from pathlib import Path

from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml.regression import IsotonicRegression
from pyspark.sql import SparkSession, Window, functions as F

BUCKET = os.environ.get("BUCKET")
RUN_ID = os.environ.get("RUN_ID")
if not BUCKET or not RUN_ID:
    raise RuntimeError("Set BUCKET and RUN_ID in the Dataproc environment.")

ROOT = f"gs://{BUCKET}/runs/{RUN_ID}"
MODEL_ROOT = f"{ROOT}/models/final_v2"
RESULTS_ROOT = f"{ROOT}/results"


def gcloud(args):
    return subprocess.run(
        ["gcloud", *args], check=True, capture_output=True, text=True
    )


def read_gcs_text(uri):
    return gcloud(["storage", "cat", uri]).stdout


config = json.loads(read_gcs_text(f"{ROOT}/config/experiment.json"))
selected_spec = json.loads(
    read_gcs_text(f"{ROOT}/artifacts/selected_model.json")
)

assert config["split_names"] == [
    "search_train", "hp_validation", "model_selection", "calibration", "test"
]
assert config["final_training_fraction"] == 1.0
assert config["calibration_method"] == "isotonic"
assert config["monotonic_projection"] == "pava"
assert config["threshold_interpolation"] == "linear"
assert selected_spec["selection_partition"] == "model_selection_2022"
assert selected_spec["final_training_partitions"] == [
    "search_train_2017_2020", "hp_validation_2021", "model_selection_2022"
]
assert selected_spec["final_training_fraction"] == 1.0

module_path = Path(tempfile.gettempdir()) / "dsa5208-rainfall" / "rainfall_modeling.py"
module_path.parent.mkdir(parents=True, exist_ok=True)
gcloud(["storage", "cp", f"{ROOT}/source/rainfall_modeling.py", str(module_path)])
sys.path.insert(0, str(module_path.parent))

from rainfall_modeling import (
    build_pipeline,
    climatology_scores,
    deterministic_sample,
    expand_thresholds,
    fit_climatology_tables,
    interpolate_thresholds,
    mark_unseen,
    monotonicity_summary,
    project_nonincreasing,
    score_model,
    threshold_metrics,
)

spark = SparkSession.builder.appName("dsa5208-rainfall-final").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", str(config["shuffle_partitions"]))
spark.conf.set("spark.sql.session.timeZone", config["timezone"])
spark.conf.set("spark.sql.ansi.enabled", "false")
actual_spark_conf = {
    name: spark.conf.get(name)
    for name in (
        "spark.sql.shuffle.partitions",
        "spark.sql.session.timeZone",
        "spark.sql.ansi.enabled",
    )
}
assert actual_spark_conf["spark.sql.shuffle.partitions"] == str(config["shuffle_partitions"])
assert actual_spark_conf["spark.sql.session.timeZone"] == config["timezone"]
assert actual_spark_conf["spark.sql.ansi.enabled"] == "false"
display({
    "selected_model": selected_spec["selected_model"],
    "spark_conf": actual_spark_conf,
})

In [ ]:
# Fit the selected model on the complete 2017–2022 training pool.
def read_final_training_partition(name):
    allowed = {"search_train", "hp_validation", "model_selection"}
    if name not in allowed:
        raise ValueError(f"Partition is not in the final training set: {name}")
    return (
        spark.read.parquet(f"{ROOT}/base_features/split={name}")
        .where(F.col("label_valid"))
        .persist(StorageLevel.DISK_ONLY)
    )


training_partitions = [
    read_final_training_partition("search_train"),
    read_final_training_partition("hp_validation"),
    read_final_training_partition("model_selection"),
]
final_training_base = reduce(
    lambda left, right: left.unionByName(right), training_partitions
).persist(StorageLevel.DISK_ONLY)
final_training_base_rows = final_training_base.count()
if final_training_base_rows == 0:
    raise ValueError("The full 2017–2022 training pool is empty.")

training_stations = final_training_base.select("station_id").distinct().persist(
    StorageLevel.DISK_ONLY
)
final_climatology_tables = fit_climatology_tables(final_training_base)
climatology_root = f"{MODEL_ROOT}/climatology"
for name, table in final_climatology_tables.items():
    table.write.mode("errorifexists").parquet(f"{climatology_root}/{name}")

selected_name = selected_spec["selected_model"]
selected_model = None
model_fit_seconds = 0.0
if selected_name == "climatology":
    if selected_spec["algorithm"] != "climatology" or selected_spec["variant"] != "baseline":
        raise ValueError("Selected baseline identity does not match the selected model.")
else:
    algorithm = selected_spec["algorithm"]
    variant = selected_spec["variant"]
    allowed_variants = {"lr": {"A"}, "rf": {"B"}, "gbt": {"A", "B"}}
    if algorithm not in allowed_variants or variant not in allowed_variants[algorithm]:
        raise ValueError(f"Unsupported selected model: {algorithm}_{variant}")
    if selected_name != f"{algorithm}_{variant}":
        raise ValueError("Selected model name does not match its algorithm and variant.")
expanded_final_training = expand_thresholds(
    final_training_base, config
).persist(StorageLevel.DISK_ONLY)
final_training_expanded_rows = final_training_base_rows * len(config["thresholds_mm"])

if selected_name != "climatology":
    started = time.perf_counter()
    selected_model = build_pipeline(
        selected_spec["algorithm"],
        selected_spec["parameters"],
        selected_spec["variant"],
        config,
    ).fit(expanded_final_training)
    model_fit_seconds = time.perf_counter() - started
    selected_model.write().save(f"{MODEL_ROOT}/selected_pipeline")

expanded_final_training.unpersist()
print({
    "selected_model": selected_name,
    "valid_2017_2022_base_rows": final_training_base_rows,
    "threshold_expanded_final_training_rows": final_training_expanded_rows,
    "final_training_fraction": 1.0,
    "model_fit_seconds": model_fit_seconds,
    "fit_used_deterministic_sample": False,
    "climatology_baseline_tables": climatology_root,
})

In [ ]:
# Fit isotonic calibration on a deterministic 10% sample of the separate 2023 partition.
calibration_base = (
    spark.read.parquet(f"{ROOT}/base_features/split=calibration")
    .where(F.col("label_valid"))
    .persist(StorageLevel.DISK_ONLY)
)
calibration_base_rows = calibration_base.count()
if calibration_base_rows == 0:
    raise ValueError("The calibration partition is empty.")

calibration_sample = deterministic_sample(
    calibration_base,
    config["calibration_sample_fraction"],
    config["sampling_seed"],
).persist(StorageLevel.DISK_ONLY)
calibration_sample_base_rows = calibration_sample.count()
if calibration_sample_base_rows == 0:
    raise ValueError("The deterministic calibration sample is empty.")

calibration_queries = mark_unseen(
    expand_thresholds(calibration_sample, config), training_stations
).persist(StorageLevel.DISK_ONLY)
if selected_name == "climatology":
    calibration_raw_scores = climatology_scores(
        calibration_queries, final_climatology_tables, config["climatology_smoothing"]
    )
else:
    calibration_raw_scores = score_model(selected_model, calibration_queries)
calibration_raw_scores = calibration_raw_scores.persist(StorageLevel.DISK_ONLY)
calibration_rows = calibration_raw_scores.count()
if calibration_rows == 0:
    raise ValueError("No valid calibration scores were produced.")

calibrator = IsotonicRegression(
    featuresCol="p",
    labelCol="label",
    predictionCol="calibrated_p",
    isotonic=True,
).fit(calibration_raw_scores)
calibrator.write().save(f"{MODEL_ROOT}/isotonic_calibrator")


def reliability_table(scores, probability_col="p"):
    bins = int(config["calibration_bins"])
    return (
        scores.withColumn(
            "_probability", F.col(probability_col)
        ).withColumn(
            "bin",
            F.least(
                F.floor(F.col("_probability") * bins), F.lit(bins - 1)
            ).cast("int"),
        )
        .groupBy("threshold_mm", "bin")
        .agg(
            F.avg("_probability").alias("predicted_probability"),
            F.avg("label").alias("observed_frequency"),
            F.count("*").alias("n"),
        )
    )


def weighted_ece(scores, probability_col="p"):
    bins = reliability_table(scores, probability_col)
    return bins.withColumn(
        "weighted_error",
        F.abs(F.col("predicted_probability") - F.col("observed_frequency"))
        * F.col("n"),
    ).agg((F.sum("weighted_error") / F.sum("n")).alias("ece")).first()["ece"]


calibration_transformed = calibrator.transform(calibration_raw_scores)
calibration_calibrated_scores = (
    calibration_transformed
    .withColumnRenamed("p", "raw_p")
    .withColumnRenamed("calibrated_p", "p")
    .persist(StorageLevel.DISK_ONLY)
)
calibration_summary = {
    "partition": "calibration_2023",
    "base_rows": calibration_base_rows,
    "sample_fraction": config["calibration_sample_fraction"],
    "sampling_seed": config["sampling_seed"],
    "sampled_base_rows": calibration_sample_base_rows,
    "expanded_calibration_rows": calibration_rows,
    "method": "isotonic",
    "always_applied": True,
    "raw_ece": weighted_ece(calibration_raw_scores),
    "calibrated_ece": weighted_ece(calibration_calibrated_scores),
}
display(calibration_summary)
calibration_calibrated_scores.unpersist()
calibration_raw_scores.unpersist()
calibration_queries.unpersist()
calibration_sample.unpersist()
calibration_base.unpersist()

In [ ]:
# Evaluate the calibrated model on 2024.
test_base = (
    spark.read.parquet(f"{ROOT}/base_features/split=test")
    .where(F.col("label_valid"))
    .persist(StorageLevel.DISK_ONLY)
)
test_base_rows = test_base.count()
if test_base_rows == 0:
    raise ValueError("The held-out evaluation partition is empty.")

test_anchor_queries = mark_unseen(
    expand_thresholds(test_base, config), training_stations
).persist(StorageLevel.DISK_ONLY)
if selected_name == "climatology":
    raw_test_scores = climatology_scores(
        test_anchor_queries, final_climatology_tables, config["climatology_smoothing"]
    )
else:
    raw_test_scores = score_model(selected_model, test_anchor_queries)
raw_test_scores = raw_test_scores.persist(StorageLevel.DISK_ONLY)

baseline_test_scores = climatology_scores(
    test_anchor_queries, final_climatology_tables, config["climatology_smoothing"]
).persist(StorageLevel.DISK_ONLY)

calibrated_test_scores = (
    calibrator.transform(raw_test_scores)
    .withColumnRenamed("p", "raw_p")
    .withColumnRenamed("calibrated_p", "p")
    .persist(StorageLevel.DISK_ONLY)
)
projected_test_scores = project_nonincreasing(
    calibrated_test_scores,
    config["thresholds_mm"],
    rainfall_integer_scale=config["rainfall_integer_scale"],
)
final_test_scores = (
    projected_test_scores
    .withColumn("calibrated_p", F.col("p"))
    .withColumn("p", F.col("projected_p"))
    .drop("projected_p")
    .persist(StorageLevel.DISK_ONLY)
)

raw_monotonicity = monotonicity_summary(
    raw_test_scores, probability_col="p", tolerance=config["monotonicity_tolerance"]
)
calibrated_monotonicity = monotonicity_summary(
    calibrated_test_scores, probability_col="p", tolerance=config["monotonicity_tolerance"]
)
final_monotonicity = monotonicity_summary(
    final_test_scores, probability_col="p", tolerance=0.0
)
if final_monotonicity["strict_violation_rate"] != 0.0:
    raise AssertionError(
        f"PAVA output is not exactly non-increasing: {final_monotonicity}"
    )

test_model_metrics = []
for model_name, scores in (
    ("raw_selected", raw_test_scores),
    ("isotonic_calibrated", calibrated_test_scores),
    ("final_pava", final_test_scores),
    ("climatology", baseline_test_scores),
):
    test_model_metrics.append({
        "model": model_name,
        "per_threshold": threshold_metrics(scores, config["thresholds_mm"], config),
        "weighted_ece": weighted_ece(scores),
        "monotonicity": (
            raw_monotonicity if model_name == "raw_selected"
            else calibrated_monotonicity if model_name == "isotonic_calibrated"
            else final_monotonicity if model_name == "final_pava"
            else monotonicity_summary(scores, probability_col="p", tolerance=0.0)
        ),
    })

final_test_scores.write.mode("errorifexists").parquet(
    f"{RESULTS_ROOT}/scores/heldout_2024_anchor_predictions"
)
for model_name, scores in (
    ("raw_selected", raw_test_scores),
    ("isotonic_calibrated", calibrated_test_scores),
    ("final_pava", final_test_scores),
    ("climatology", baseline_test_scores),
):
    reliability_table(scores).write.mode("errorifexists").json(
        f"{RESULTS_ROOT}/reliability/{model_name}"
    )

# Arbitrary in-domain threshold queries use linear interpolation on the
# projected anchor curve. Out-of-domain values are rejected by the helper.
unseen_thresholds = [float(value) for value in config["unseen_thresholds_mm"]]
domain_low, domain_high = map(float, config["threshold_domain_mm"])
if any(value < domain_low or value > domain_high for value in unseen_thresholds):
    raise ValueError("Configured interpolation thresholds exceed the allowed domain.")
unseen_queries = mark_unseen(
    expand_thresholds(test_base, config, unseen_thresholds), training_stations
)
unseen_scores = interpolate_thresholds(
    projected_test_scores,
    unseen_queries,
    config["thresholds_mm"],
    output_col="p",
    requested_thresholds=unseen_thresholds,
).persist(StorageLevel.DISK_ONLY)
unseen_metrics = threshold_metrics(unseen_scores, unseen_thresholds, config)
unseen_monotonicity = monotonicity_summary(
    unseen_scores, probability_col="p", tolerance=0.0
)
if unseen_monotonicity["strict_violation_rate"] != 0.0:
    raise AssertionError(
        f"Interpolated probabilities are not monotone: {unseen_monotonicity}"
    )

# Paired weekly-block bootstrap compares the final constrained predictor with
# full-training climatology on the same station/time/threshold cohort.
keys = ["station_id", "epoch", "threshold_mm"]
selected_losses = final_test_scores.select(
    *keys,
    "local_date",
    "label",
    F.pow(F.col("p") - F.col("label"), 2).alias("selected_loss"),
)
baseline_losses = baseline_test_scores.select(
    *keys,
    F.col("label").alias("baseline_label"),
    F.pow(F.col("p") - F.col("label"), 2).alias("baseline_loss"),
)
paired_losses = selected_losses.join(baseline_losses, keys, "inner").persist(
    StorageLevel.DISK_ONLY
)
paired_count = paired_losses.count()
if paired_losses.where("label != baseline_label").limit(1).count():
    raise ValueError("Paired test labels differ between predictor and climatology.")
if paired_count != final_test_scores.count() or paired_count != baseline_test_scores.count():
    raise ValueError("Final predictor and baseline test cohorts do not match.")

weekly = (
    paired_losses.withColumn("block", F.date_trunc("week", "local_date"))
    .groupBy("block")
    .agg(
        F.sum(F.col("selected_loss") - F.col("baseline_loss")).alias("difference_sum"),
        F.count("*").alias("n"),
    )
    .orderBy("block")
    .collect()
)
if len(weekly) > 60:
    raise ValueError("Expected at most 60 calendar-week blocks in the evaluation period.")
paired_losses.unpersist()
difference = sum(row["difference_sum"] for row in weekly) / sum(row["n"] for row in weekly)
if len(weekly) < 2:
    ci95 = None
else:
    rng = random.Random(config["sampling_seed"])
    bootstrap = []
    for _ in range(config["bootstrap_replicates"]):
        sample = [weekly[rng.randrange(len(weekly))] for _ in weekly]
        bootstrap.append(
            sum(row["difference_sum"] for row in sample)
            / sum(row["n"] for row in sample)
        )
    bootstrap.sort()
    ci95 = [
        bootstrap[int(0.025 * (len(bootstrap) - 1))],
        bootstrap[int(0.975 * (len(bootstrap) - 1))],
    ]
paired_weekly_brier = {
    "comparison": "final PAVA predictor minus climatology",
    "difference": difference,
    "ci95": ci95,
    "week_blocks": len(weekly),
    "bootstrap_replicates": config["bootstrap_replicates"] if ci95 else 0,
}

evaluation_summary = {
    "selected_model": selected_name,
    "heldout_evaluation_period": 2024,
    "valid_test_base_rows": test_base_rows,
    "final_training_base_rows": final_training_base_rows,
    "final_training_expanded_rows": final_training_expanded_rows,
    "model_fit_seconds": model_fit_seconds,
    "calibration": calibration_summary,
    "metrics": test_model_metrics,
    "unseen_thresholds": {
        "method": "linear interpolation after PAVA",
        "domain_mm": [domain_low, domain_high],
        "thresholds_mm": unseen_thresholds,
        "metrics": unseen_metrics,
        "monotonicity": unseen_monotonicity,
    },
    "paired_weekly_brier": paired_weekly_brier,
    "final_monotonicity": final_monotonicity,
    "artifacts": {
        "selected_pipeline": (
            f"{MODEL_ROOT}/selected_pipeline" if selected_model is not None else None
        ),
        "climatology": climatology_root,
        "calibrator": f"{MODEL_ROOT}/isotonic_calibrator",
        "selected_model_spec": f"{ROOT}/artifacts/selected_model.json",
        "anchor_scores": f"{RESULTS_ROOT}/scores/heldout_2024_anchor_predictions",
    },
}
summary_path = Path(tempfile.gettempdir()) / f"dsa5208-{RUN_ID}-final-summary.json"
summary_path.write_text(
    json.dumps(evaluation_summary, indent=2, sort_keys=True, allow_nan=False) + "\n",
    encoding="utf-8",
)
gcloud([
    "storage", "cp", str(summary_path),
    f"{RESULTS_ROOT}/final_evaluation_summary.json",
])
summary_path.unlink(missing_ok=True)
display(evaluation_summary)

unseen_scores.unpersist()
final_test_scores.unpersist()
calibrated_test_scores.unpersist()
raw_test_scores.unpersist()
baseline_test_scores.unpersist()
test_anchor_queries.unpersist()
test_base.unpersist()
for frame in training_partitions:
    frame.unpersist()
training_stations.unpersist()
final_training_base.unpersist()
spark.stop()